# 🚀 Big Data Pipeline: Amazon Sale Report
**Teknologi:** PySpark | MySQL | MongoDB  
**Domain:** E-commerce Analytics  
**Dataset:** Amazon Sale Report (~128.000 baris)

---
## Pipeline Overview
```
CSV File → PySpark Ingestion → Cleaning → Processing → MySQL + MongoDB → Integration → Analisis
```

---
## 📋 DOKUMENTASI DATASET & DATABASE

### 📊 Kolom-Kolom Dataset Asli (CSV)

Dataset `Amazon Sale Report.csv` memiliki **24 kolom** berikut:

| # | Nama Kolom | Tipe Data | Deskripsi |
|---|-----------|-----------|----------|
| 1 | `row_index` | String | ❌ Index baris (tidak berguna, akan di-drop) |
| 2 | `order_id` | String | ✅ ID unik setiap order |
| 3 | `date` | String→Date | ✅ Tanggal order (format M/d/yyyy) |
| 4 | `status` | String | ✅ Status order detail (Shipped, Cancelled, dll) |
| 5 | `fulfilment` | String | ✅ Tipe fulfilment (Amazon/Merchant) |
| 6 | `sales_channel` | String | ✅ Channel penjualan |
| 7 | `ship_service_level` | String | ✅ Level layanan pengiriman |
| 8 | `style` | String | ✅ Style produk |
| 9 | `sku` | String | ✅ Stock Keeping Unit |
| 10 | `category` | String | ✅ Kategori produk |
| 11 | `size` | String | ✅ Ukuran produk |
| 12 | `asin` | String | ✅ Amazon Standard Identification Number |
| 13 | `courier_status` | String | ✅ Status kurir (banyak null) |
| 14 | `qty` | Integer | ✅ Jumlah item |
| 15 | `currency` | String | ✅ Mata uang (INR) |
| 16 | `amount` | Double | ✅ Harga satuan |
| 17 | `ship_city` | String | ✅ Kota tujuan pengiriman |
| 18 | `ship_state` | String | ✅ Negara bagian tujuan |
| 19 | `ship_postal_code` | String | ✅ Kode pos tujuan |
| 20 | `ship_country` | String | ✅ Negara tujuan |
| 21 | `promotion_ids` | String | ✅ ID promosi (banyak null) |
| 22 | `b2b` | Boolean | ✅ Apakah order B2B (Business-to-Business) |
| 23 | `fulfilled_by` | String | ✅ Pihak yang memenuhi order (banyak null) |
| 24 | `unused_col` | String | ❌ Kolom kosong (tidak berguna, akan di-drop) |

### 🔄 Kolom Tambahan Hasil Cleaning/Processing

| Nama Kolom Baru | Dibuat dari | Deskripsi |
|----------------|------------|----------|
| `year` | `date` | Tahun dari tanggal order |
| `month` | `date` | Bulan (angka) |
| `month_name` | `date` | Nama bulan |
| `status_group` | `status` | Normalisasi status: Shipped/Cancelled/Pending/Other |
| `revenue` | `qty × amount` | Total pendapatan per order |
| `has_promotion` | `promotion_ids` | Flag Boolean ada/tidak promosi |

---

### 🗄️ Peta Kolom Dataset → Database

```
╔══════════════════════════════════════════════════════════════════════════════════╗
║                     AMAZON SALE REPORT DATASET (24 kolom)                      ║
╠══════════════════════════════════════════════════════════════════════════════════╣
║  order_id │ date │ status │ fulfilment │ sales_channel │ ship_service_level    ║
║  style │ sku │ category │ size │ asin │ courier_status │ qty │ currency        ║
║  amount │ ship_city │ ship_state │ ship_postal_code │ ship_country             ║
║  promotion_ids │ b2b │ fulfilled_by                                             ║
║  [DERIVED: year, month, month_name, status_group, revenue, has_promotion]       ║
╚══════════════════════════════════════════════════════════════════════════════════╝
                    │                               │
         ┌──────────┘                               └──────────┐
         ▼                                                     ▼
╔══════════════════════════╗                    ╔═══════════════════════════════╗
║        MySQL             ║                    ║          MongoDB              ║
║  (Structured/Relational) ║                    ║     (Semi-Structured/NoSQL)   ║
╠══════════════════════════╣                    ╠═══════════════════════════════╣
║  TABLE: orders           ║                    ║  COLLECTION: orders_detail    ║
║  ├─ id (PK)              ║                    ║  ├─ order_id                  ║
║  ├─ order_id (UNIQUE)    ║                    ║  ├─ date (string)             ║
║  ├─ date                 ║                    ║  ├─ status                    ║
║  ├─ year                 ║                    ║  ├─ status_group              ║
║  ├─ month                ║                    ║  ├─ fulfilment                ║
║  ├─ month_name           ║                    ║  ├─ sales_channel             ║
║  ├─ status               ║                    ║  ├─ category                  ║
║  ├─ status_group         ║                    ║  ├─ size, sku, asin           ║
║  ├─ fulfilment           ║                    ║  ├─ qty, amount, revenue      ║
║  ├─ sales_channel        ║                    ║  ├─ ship_city, ship_state     ║
║  ├─ category             ║                    ║  ├─ b2b, has_promotion        ║
║  ├─ size, sku, asin      ║                    ║  ├─ promotion_ids (ARRAY)     ║
║  ├─ qty, amount          ║                    ║  └─ _source                   ║
║  ├─ revenue              ║                    ╠═══════════════════════════════╣
║  ├─ currency             ║                    ║  COLLECTION: category_insights║
║  ├─ ship_city            ║                    ║  ├─ category                  ║
║  ├─ ship_state           ║                    ║  ├─ total_orders              ║
║  ├─ ship_postal_code     ║                    ║  ├─ total_qty_sold            ║
║  ├─ ship_country         ║                    ║  ├─ total_revenue             ║
║  ├─ b2b                  ║                    ║  ├─ avg_price                 ║
║  ├─ has_promotion        ║                    ║  ├─ price_range (OBJECT)      ║
║  ├─ courier_status       ║                    ║  │   ├─ min                   ║
║  └─ fulfilled_by         ║                    ║  │   └─ max                   ║
╠══════════════════════════╣                    ║  └─ sizes_breakdown (ARRAY)   ║
║  TABLE: monthly_summary  ║                    ╚═══════════════════════════════╝
║  ├─ id (PK)              ║
║  ├─ year                 ║
║  ├─ month                ║
║  ├─ month_name           ║
║  ├─ total_orders         ║
║  ├─ total_revenue        ║
║  ├─ total_qty            ║
║  └─ unique_categories    ║
╠══════════════════════════╣
║  TABLE: category_stats   ║
║  ├─ id (PK)              ║
║  ├─ category (UNIQUE)    ║
║  ├─ total_orders         ║
║  ├─ total_qty_sold       ║
║  ├─ total_revenue        ║
║  ├─ avg_price            ║
║  ├─ max_price            ║
║  └─ min_price            ║
╚══════════════════════════╝

KOLOM YANG TIDAK DISIMPAN (di-drop):
  ❌ row_index   → Tidak bermakna, hanya index baris CSV
  ❌ unused_col  → Kolom kosong tanpa nilai
  ❌ style       → Tidak disertakan (redundan dengan sku)
  ❌ ship_service_level → Tidak disertakan di tabel orders MySQL
```

### 🔑 Alasan Pemisahan MySQL vs MongoDB

| Aspek | MySQL | MongoDB |
|-------|-------|--------|
| **Tipe Data** | Flat/Tabular | Nested/Array |
| **Kegunaan** | Transaksi, JOIN, Reporting | Dokumen lengkap, Analytics |
| **Contoh** | Laporan penjualan, Summary bulanan | Detail order dengan promo array, Insight kategori |
| **Kelebihan** | ACID, Relasi, Index efisien | Flexibel schema, Nested data |

> 💡 **Catatan:** `promotion_ids` di MongoDB disimpan sebagai **ARRAY** (bukan string), sedangkan di MySQL disimpan sebagai string karena batasan tipe data relasional.

---
## 📦 0. Install Dependencies

In [1]:
import subprocess
import sys

packages = [
    "pyspark",
    "pymongo",
    "mysql-connector-python",
    "findspark"
]

print("📦 Installing required packages...\n")
for package in packages:
    try:
        __import__(package.replace("-", "_"))
        print(f"✅ {package} already installed")
    except ImportError:
        print(f"⏳ Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"✅ {package} installed")

print("\n✅ All dependencies ready!")
print("\n💡 Note: Visualisasi dashboard menggunakan app.py (Streamlit) — jalankan terpisah:")
print("   streamlit run app.py")

📦 Installing required packages...

✅ pyspark already installed
✅ pymongo already installed
⏳ Installing mysql-connector-python...
✅ mysql-connector-python installed
✅ findspark already installed

✅ All dependencies ready!

💡 Note: Visualisasi dashboard menggunakan app.py (Streamlit) — jalankan terpisah:
   streamlit run app.py


---
## 🔧 1. Setup & Konfigurasi

In [2]:
import os
import platform
from pathlib import Path

# ==================== CSV PATH ====================
CSV_PATH = r"C:\Users\lenov\BIDAT\Amazon Sale Report.csv"

if not os.path.exists(CSV_PATH):
    print(f"❌ ERROR: File tidak ditemukan di {CSV_PATH}")
else:
    file_size_mb = os.path.getsize(CSV_PATH) / (1024 * 1024)
    print(f"✅ CSV File ditemukan: {file_size_mb:.1f} MB")

# ==================== MySQL CONFIG ====================
MYSQL_HOST     = "localhost"
MYSQL_PORT     = 3306
MYSQL_USER     = "root"
MYSQL_PASSWORD = ""  # Sesuaikan dengan password MySQL Anda
MYSQL_DATABASE = "final_bidat"

POSSIBLE_JDBC_PATHS = [
    r"C:\Users\lenov\BIDAT\mysql-connector-j-9.7.0\mysql-connector-j-9.7.0.jar",
]

MYSQL_JDBC_JAR = None
for path in POSSIBLE_JDBC_PATHS:
    if os.path.exists(path):
        MYSQL_JDBC_JAR = path
        break

if not MYSQL_JDBC_JAR:
    print("WARNING: MySQL JDBC Driver tidak ditemukan")
else:
    print(f"MySQL JDBC Driver: {MYSQL_JDBC_JAR}")

MYSQL_JDBC_URL = f"jdbc:mysql://{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}?allowPublicKeyRetrieval=true&useSSL=false"

# ==================== MongoDB CONFIG ====================
MONGO_URI      = "mongodb://localhost:27017/"
MONGO_DATABASE = "final_bidat"

print(f"\n{'='*60}")
print(f" Konfigurasi berhasil dimuat")
print(f"   OS      : {platform.system()}")
print(f"   CSV     : {CSV_PATH}")
print(f"   MySQL   : {MYSQL_HOST}:{MYSQL_PORT}")
print(f"   MongoDB : {MONGO_URI}")
print(f"{'='*60}")

✅ CSV File ditemukan: 65.5 MB
MySQL JDBC Driver: C:\Users\lenov\BIDAT\mysql-connector-j-9.7.0\mysql-connector-j-9.7.0.jar

 Konfigurasi berhasil dimuat
   OS      : Windows
   CSV     : C:\Users\lenov\BIDAT\Amazon Sale Report.csv
   MySQL   : localhost:3306
   MongoDB : mongodb://localhost:27017/


In [3]:
import os
import subprocess

JAVA_HOME = r"C:\Program Files\Java\jdk-17.0.11"
os.environ["JAVA_HOME"] = JAVA_HOME
os.environ["PATH"] = JAVA_HOME + r"\bin;" + os.environ["PATH"]
os.environ["HADOOP_HOME"] = r"C:\hadoop-3.3.6"
os.environ["PATH"] += r";C:\hadoop-3.3.6\bin"

result = subprocess.run(["java", "-version"], capture_output=True, text=True)
print(f"JAVA_HOME  : {os.environ['JAVA_HOME']}")
print(f"HADOOP_HOME: {os.environ['HADOOP_HOME']}")
print(f"Java Version:\n{result.stderr}")

JAVA_HOME  : C:\Program Files\Java\jdk-17.0.11
HADOOP_HOME: C:\hadoop-3.3.6
Java Version:
java version "17.0.11" 2024-04-16 LTS
Java(TM) SE Runtime Environment (build 17.0.11+7-LTS-207)
Java HotSpot(TM) 64-Bit Server VM (build 17.0.11+7-LTS-207, mixed mode, sharing)



---
## 🚀 2. Inisialisasi PySpark Session

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import os, sys, warnings

warnings.filterwarnings('ignore')

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("🚀 Initializing PySpark Session...\n")

builder = (
    SparkSession.builder
    .appName("AmazonSalesPipeline")

    # FIX worker crash
    .master("local[2]")

    # Memory
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")

    # Kurangi parallel task
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.default.parallelism", "2")

    # Disable Arrow
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")

    # Stabilkan Python worker
    .config("spark.python.worker.faulthandler.enabled", "true")

    # Host
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
)

if MYSQL_JDBC_JAR and os.path.exists(MYSQL_JDBC_JAR):
    builder = builder.config(
        "spark.jars",
        MYSQL_JDBC_JAR
    )

spark = builder.getOrCreate()

spark.sparkContext.setCheckpointDir(
    "checkpoint"
)

spark.sparkContext.setLogLevel("WARN")

print(f"✅ Spark version : {spark.version}")
print(f"✅ SparkSession  : ready!")

🚀 Initializing PySpark Session...

✅ Spark version : 3.5.3
✅ SparkSession  : ready!


---
## 📥 3. Data Ingestion — Load CSV dengan PySpark

In [5]:
# ================================================
# STEP 1: Load CSV menggunakan PySpark
# Schema eksplisit sesuai 24 kolom dataset
# ================================================

# Schema (sama seperti sebelumnya)
schema = StructType([
    StructField("row_index",          StringType(),  True),
    StructField("order_id",           StringType(),  True),
    StructField("date",               StringType(),  True),
    StructField("status",             StringType(),  True),
    StructField("fulfilment",         StringType(),  True),
    StructField("sales_channel",      StringType(),  True),
    StructField("ship_service_level", StringType(),  True),
    StructField("style",              StringType(),  True),
    StructField("sku",                StringType(),  True),
    StructField("category",           StringType(),  True),
    StructField("size",               StringType(),  True),
    StructField("asin",               StringType(),  True),
    StructField("courier_status",     StringType(),  True),
    StructField("qty",                IntegerType(), True),
    StructField("currency",           StringType(),  True),
    StructField("amount",             DoubleType(),  True),
    StructField("ship_city",          StringType(),  True),
    StructField("ship_state",         StringType(),  True),
    StructField("ship_postal_code",   StringType(),  True),
    StructField("ship_country",       StringType(),  True),
    StructField("promotion_ids",      StringType(),  True),
    StructField("b2b",                BooleanType(), True),
    StructField("fulfilled_by",       StringType(),  True),
    StructField("unused_col",         StringType(),  True),
])

df_raw = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .schema(schema)
    .csv(CSV_PATH)
)

# ─── FIX: Normalisasi format tanggal dengan coalesce multi-format ─────────────
# Dataset Amazon Sale Report menggunakan format "04-30-22" (MM-dd-yy),
# NAMUN beberapa baris bisa memiliki format berbeda (M/d/yyyy, dd-MM-yy, dll).
# Dengan coalesce, Spark mencoba tiap format secara berurutan dan memakai
# yang pertama berhasil — sehingga tidak ada tanggal yang jatuh jadi null
# hanya karena perbedaan format.
# PENTING: Konversi ke string dihapus dari step ini.
#          Parsing DateType final dilakukan di step Cleaning (cell berikutnya).
df_raw = df_raw.withColumn(
    "date",
    F.coalesce(
        F.to_date(F.col("date"), "MM-dd-yy"),   # format utama: 04-30-22
        F.to_date(F.col("date"), "M/d/yyyy"),    # fallback 1 : 4/30/2022
        F.to_date(F.col("date"), "dd-MM-yy"),    # fallback 2 : 30-04-22
        F.to_date(F.col("date"), "yyyy-MM-dd"),  # fallback 3 : 2022-04-30
    )
)
# Setelah coalesce, kolom "date" sudah bertipe DateType — tidak perlu
# dikonversi lagi di step Cleaning, cukup langsung ekstrak year/month/month_name.

# Bersihkan ship_postal_code: hilangkan ".0" di akhir
df_raw = df_raw.withColumn(
    "ship_postal_code",
    F.regexp_replace(F.col("ship_postal_code"), r"\.0$", "")
)

# ─── Validasi: cek berapa baris yang date-nya masih null setelah coalesce ─────
null_date_count = df_raw.filter(F.col("date").isNull()).count()
total_raw = df_raw.count()
print(f"✅ Data berhasil dimuat")
print(f"   Total baris        : {total_raw:,}")
print(f"   Total kolom        : {len(df_raw.columns)}")
print(f"   Baris date = null  : {null_date_count:,}  ", end="")
print("← idealnya 0" if null_date_count == 0 else "← ⚠️  ada tanggal yang tidak terbaca")
print()

df_raw.show(3, truncate=True)


✅ Data berhasil dimuat
   Total baris        : 128,975
   Total kolom        : 24
   Baris date = null  : 0  ← idealnya 0

+---------+-------------------+----------+--------------------+----------+-------------+------------------+-------+---------------+--------+----+----------+--------------+---+--------+------+-----------+-----------+----------------+------------+--------------------+-----+------------+----------+
|row_index|           order_id|      date|              status|fulfilment|sales_channel|ship_service_level|  style|            sku|category|size|      asin|courier_status|qty|currency|amount|  ship_city| ship_state|ship_postal_code|ship_country|       promotion_ids|  b2b|fulfilled_by|unused_col|
+---------+-------------------+----------+--------------------+----------+-------------+------------------+-------+---------------+--------+----+----------+--------------+---+--------+------+-----------+-----------+----------------+------------+--------------------+-----+-----------

---
## 🧹 4. Data Cleaning & Preprocessing

| Kolom | Masalah | Solusi | Justifikasi |
|---|---|---|---|
| `row_index`, `unused_col` | Kolom tidak berguna | Drop | Tidak memiliki nilai analitik |
| `date` | Format string | Konversi ke DateType | Diperlukan untuk ekstraksi year/month |
| `style` | Redundan dengan `sku` | Drop | Informasi sudah terwakili di `sku` |
| `ship_service_level` | Tidak digunakan di analitik | Drop dari MySQL | Tidak relevan untuk query bisnis utama |
| `courier_status` | 6.872 null (~5.4%) | Isi 'Unknown' | Null-rate rendah, nilai bisa diisi logis |
| `fulfilled_by` | 89.698 null (~70%) | Isi 'Unknown' | Null karena Amazon auto-fulfil tidak mengisi field ini |
| `promotion_ids` | 49.153 null (~38%) | Isi 'None' | Null berarti tidak ada promosi — penggantian logis |
| `currency` | 7.795 null | Isi 'INR' | Seluruh dataset dalam mata uang INR |
| `amount` | 7.795 null | Drop baris | Tidak ada amount = data transaksi tidak valid |
| `ship_city/state/postal/country` | 33 null | Drop baris | 33 baris (<0.03%) — aman dihapus, tidak pengaruh signifikan |
| `qty` | Nilai negatif | Filter qty >= 0 | Qty negatif tidak masuk akal secara bisnis |
| `order_id` | Duplikat | dropDuplicates | Satu order hanya boleh muncul sekali |
| `sales_channel` | Trailing whitespace | Trim | Normalisasi untuk konsistensi groupBy |
| `status` | Banyak variant teks | Normalisasi → status_group | Dikelompokkan: Shipped/Cancelled/Pending/Other |


In [6]:
# ================================================
# STEP 2: Data Cleaning Pipeline
# ================================================

# --- 2a. Cek missing values ---
print("=== Missing Values per Kolom ===")
total = df_raw.count()
for col_name in df_raw.columns:
    null_count = df_raw.filter(F.col(col_name).isNull()).count()
    if null_count > 0:
        pct = null_count / total * 100
        print(f"  {col_name:<25}: {null_count:>6,} null ({pct:.1f}%)")

=== Missing Values per Kolom ===
  courier_status           :  6,872 null (5.3%)
  currency                 :  7,795 null (6.0%)
  amount                   :  7,795 null (6.0%)
  ship_city                :     33 null (0.0%)
  ship_state               :     33 null (0.0%)
  ship_postal_code         :     33 null (0.0%)
  ship_country             :     33 null (0.0%)
  promotion_ids            : 49,153 null (38.1%)
  fulfilled_by             : 89,698 null (69.5%)
  unused_col               : 49,050 null (38.0%)


In [7]:
# --- 2b. Cleaning pipeline ---

df_clean = (
    df_raw
    # 1. Hapus kolom tidak berguna
    .drop("row_index", "unused_col")
    # 2. Trim whitespace
    .withColumn("sales_channel", F.trim(F.col("sales_channel")))
    .withColumn("ship_city",     F.trim(F.upper(F.col("ship_city"))))
    .withColumn("ship_state",    F.trim(F.upper(F.col("ship_state"))))
    .withColumn("status",        F.trim(F.col("status")))
    .withColumn("category",      F.trim(F.col("category")))
    # ─── FIX: Kolom "date" sudah DateType dari step Ingestion (cell sebelumnya).
    #          Tidak perlu to_date() lagi — langsung ekstrak komponen waktu. ─────
    # 3. Ekstrak komponen waktu dari date (sudah DateType, tidak perlu konversi)
    .withColumn("year",       F.year(F.col("date")))
    .withColumn("month",      F.month(F.col("date")))
    .withColumn("month_name", F.date_format(F.col("date"), "MMMM"))
    # 4. Normalisasi status → status_group (4 kategori utama)
    .withColumn("status_group",
        F.when(F.col("status").startswith("Shipped"), "Shipped")
         .when(F.col("status") == "Cancelled", "Cancelled")
         .when(F.col("status") == "Pending", "Pending")
         .otherwise("Other")
    )
    # ─── Encoding Kategorikal: status_group → status_encoded ─────────────────
    # Menggunakan label encoding manual (bukan StringIndexer) agar urutan
    # kategori bermakna dan konsisten di semua platform:
    #   0 = Shipped   (status sukses / mayoritas data)
    #   1 = Cancelled (status gagal / dibatalkan)
    #   2 = Pending   (status menunggu)
    #   3 = Other     (status lainnya)
    # Pendekatan ini mudah diinterpretasikan dan tidak memerlukan fitting.
    .withColumn("status_encoded",
        F.when(F.col("status_group") == "Shipped",   0)
         .when(F.col("status_group") == "Cancelled", 1)
         .when(F.col("status_group") == "Pending",   2)
         .otherwise(3)  # Other
    )
    # ─────────────────────────────────────────────────────────────────────────
    # 5. Isi null dengan nilai yang logis dan dapat dipertanggungjawabkan:
    #    - courier_status: null karena kurir belum update → 'Unknown'
    #    - fulfilled_by: null karena Amazon auto-fulfil → 'Unknown'
    #    - promotion_ids: null berarti tidak ada promosi → 'None'
    #    - currency: seluruh dataset INR → 'INR'
    .fillna({"courier_status": "Unknown", "fulfilled_by": "Unknown",
             "promotion_ids": "None", "currency": "INR"})
    # 6. Drop baris null kritis:
    #    - date: tanggal null tidak bisa dianalitik (jumlah kecil)
    #    - amount: transaksi tanpa nilai = data tidak valid (~7.8k baris)
    #    - ship_city/state/country: alamat null = data tidak valid (~33 baris)
    .dropna(subset=["date", "amount", "ship_city", "ship_state", "ship_country"])
    # 7. Hapus duplikat berdasarkan order_id
    .dropDuplicates(["order_id"])
    # 8. Filter nilai tidak logis:
    #    qty < 0: tidak mungkin terjadi dalam konteks penjualan nyata
    #    amount < 0: tidak mungkin (harga tidak boleh negatif)
    .filter(F.col("qty") >= 0)
    .filter(F.col("amount") >= 0)
    # 9. Tambah kolom derived
    .withColumn("revenue", F.round(F.col("qty") * F.col("amount"), 2))
    .withColumn("has_promotion",
        F.when(F.col("promotion_ids") != "None", True).otherwise(False)
    )
)

df_clean.cache()
clean_count = df_clean.count()
print(f"✅ Cleaning selesai!")
print(f"   Baris sebelum : {total:,}")
print(f"   Baris sesudah : {clean_count:,}")
print(f"   Dihapus       : {total - clean_count:,} ({(total - clean_count)/total*100:.1f}%)")
print()
print("   📋 Justifikasi penghapusan baris:")
print("   - amount null  : ~7.795 baris  → transaksi tanpa nilai tidak valid")
print("   - ship_* null  : ~33 baris     → alamat tidak valid")
print("   - date null    : sisa kecil    → tidak bisa dianalitik")
print("   - qty < 0      : sangat kecil  → tidak logis secara bisnis")
print("   - duplikat     : varies        → satu order hanya boleh muncul sekali")
print()
print("   📌 Kolom status_encoded (label encoding):")
print("      0 = Shipped | 1 = Cancelled | 2 = Pending | 3 = Other")

# ─── Validasi tanggal setelah cleaning ────────────────────────────────────────
null_after = df_clean.filter(F.col("date").isNull()).count()
print(f"\n   Null date (post-clean) : {null_after} ← idealnya 0")

df_clean.printSchema()


✅ Cleaning selesai!
   Baris sebelum : 128,975
   Baris sesudah : 113,004
   Dihapus       : 15,971 (12.4%)

   📋 Justifikasi penghapusan baris:
   - amount null  : ~7.795 baris  → transaksi tanpa nilai tidak valid
   - ship_* null  : ~33 baris     → alamat tidak valid
   - date null    : sisa kecil    → tidak bisa dianalitik
   - qty < 0      : sangat kecil  → tidak logis secara bisnis
   - duplikat     : varies        → satu order hanya boleh muncul sekali

   📌 Kolom status_encoded (label encoding):
      0 = Shipped | 1 = Cancelled | 2 = Pending | 3 = Other

   Null date (post-clean) : 0 ← idealnya 0
root
 |-- order_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- fulfilment: string (nullable = true)
 |-- sales_channel: string (nullable = true)
 |-- ship_service_level: string (nullable = true)
 |-- style: string (nullable = true)
 |-- sku: string (nullable = true)
 |-- category: string (nullable = true)
 |-- size: string (nul

---
## 💾 4b. Export Hasil Cleaning ke CSV

Simpan `df_clean` ke file CSV lokal sebagai output pipeline cleaning.


In [8]:
# ================================================
# STEP 4b: Export df_clean ke CSV
# ================================================

import os

# Path output CSV
CSV_OUTPUT_PATH = r"C:\Users\lenov\BIDAT\amazon_cleaned.csv"

# PySpark menulis ke folder — kita gabungkan dengan coalesce(1) lalu rename
TEMP_SPARK_DIR = r"C:\Users\lenov\BIDAT\amazon_cleaned_spark_tmp"

print("💾 Menyimpan hasil cleaning ke CSV...")

# Tulis CSV menggunakan PySpark (1 partisi supaya jadi 1 file)
(
    df_clean
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .option("encoding", "UTF-8")
    .csv(TEMP_SPARK_DIR)
)

# Rename part file → amazon_cleaned.csv
import glob
part_files = glob.glob(os.path.join(TEMP_SPARK_DIR, "part-*.csv"))
if part_files:
    import shutil
    shutil.move(part_files[0], CSV_OUTPUT_PATH)
    shutil.rmtree(TEMP_SPARK_DIR, ignore_errors=True)
    file_size_mb = os.path.getsize(CSV_OUTPUT_PATH) / (1024 * 1024)
    print(f"✅ CSV tersimpan: {CSV_OUTPUT_PATH}")
    print(f"   Ukuran file   : {file_size_mb:.1f} MB")
    print(f"   Jumlah baris  : {clean_count:,} (+ 1 header)")
    print()
    print("📋 Kolom yang tersimpan di CSV:")
    print("   Semua kolom df_clean termasuk:")
    print("   - status_encoded (hasil label encoding)")
    print("   - revenue, has_promotion (derived columns)")
    print("   - year, month, month_name (time features)")
else:
    print("❌ Gagal menemukan part file dari PySpark output")
    print(f"   Cek folder: {TEMP_SPARK_DIR}")


💾 Menyimpan hasil cleaning ke CSV...
✅ CSV tersimpan: C:\Users\lenov\BIDAT\amazon_cleaned.csv
   Ukuran file   : 62.9 MB
   Jumlah baris  : 113,004 (+ 1 header)

📋 Kolom yang tersimpan di CSV:
   Semua kolom df_clean termasuk:
   - status_encoded (hasil label encoding)
   - revenue, has_promotion (derived columns)
   - year, month, month_name (time features)


---
## ⚙️ 5. Data Processing — Analisis dengan Spark

In [9]:
# --- 3a. Statistik per Kategori Produk ---
df_by_category = (
    df_clean
    .groupBy("category")
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("qty").alias("total_qty_sold"),
        F.sum("revenue").alias("total_revenue"),
        F.avg("amount").alias("avg_price"),
        F.max("amount").alias("max_price"),
        F.min("amount").alias("min_price"),
    )
    .withColumn("total_revenue", F.round("total_revenue", 2))
    .withColumn("avg_price",     F.round("avg_price", 2))
    .orderBy(F.desc("total_revenue"))
)
print("=== 📦 Statistik per Kategori ===")
df_by_category.show(20, truncate=False)

=== 📦 Statistik per Kategori ===
+-------------+------------+--------------+-------------+---------+---------+---------+
|category     |total_orders|total_qty_sold|total_revenue|avg_price|max_price|min_price|
+-------------+------------+--------------+-------------+---------+---------+---------+
|Set          |44123       |42433         |3.5516449E7  |832.72   |5495.0   |0.0      |
|kurta        |42957       |41358         |1.9016764E7  |456.35   |2796.0   |0.0      |
|Western Dress|13951       |13220         |1.0137591E7  |762.12   |2860.0   |0.0      |
|Top          |9585        |9333          |4928500.0    |524.85   |1764.0   |0.0      |
|Ethnic Dress |1020        |982           |706521.0     |718.94   |1449.0   |0.0      |
|Blouse       |846         |811           |425041.0     |521.56   |1266.66  |0.0      |
|Bottom       |387         |367           |131704.0     |356.13   |729.0    |0.0      |
|Saree        |134         |130           |108339.0     |805.06   |2058.0   |0.0      |

In [10]:
# --- 3b. Tren Penjualan Bulanan ---
df_monthly = (
    df_clean
    .filter(F.col("status_group") == "Shipped")
    .groupBy("year", "month", "month_name")
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("revenue").alias("total_revenue"),
        F.sum("qty").alias("total_qty"),
        F.countDistinct("category").alias("unique_categories"),
    )
    .withColumn("total_revenue", F.round("total_revenue", 2))
    .orderBy("year", "month")
)
print("=== 📅 Tren Penjualan Bulanan ===")
df_monthly.show(20, truncate=False)

=== 📅 Tren Penjualan Bulanan ===
+----+-----+----------+------------+-------------+---------+-----------------+
|year|month|month_name|total_orders|total_revenue|total_qty|unique_categories|
+----+-----+----------+------------+-------------+---------+-----------------+
|2022|3    |March     |140         |89098.0      |140      |5                |
|2022|4    |April     |39071       |2.4705594E7  |39198    |8                |
|2022|5    |May       |33680       |2.2532925E7  |33786    |8                |
|2022|6    |June      |29311       |1.9593807E7  |29417    |9                |
+----+-----+----------+------------+-------------+---------+-----------------+



In [11]:
# --- 3c. Distribusi Status Order ---
df_status = (
    df_clean
    .groupBy("status_group")
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("revenue").alias("total_revenue"),
    )
    .withColumn("pct_orders",
        F.round(F.col("total_orders") / F.sum("total_orders").over(Window.orderBy(F.lit(1))) * 100, 2)
    )
    .withColumn("total_revenue", F.round("total_revenue", 2))
    .orderBy(F.desc("total_orders"))
)
print("=== 📊 Distribusi Status Order ===")
df_status.show()

=== 📊 Distribusi Status Order ===
+------------+------------+-------------+----------+
|status_group|total_orders|total_revenue|pct_orders|
+------------+------------+-------------+----------+
|     Shipped|      102202|  6.6921424E7|     90.44|
|   Cancelled|        9956|    3482296.0|      8.81|
|     Pending|         584|     387144.0|      0.52|
|       Other|         262|     180350.0|      0.23|
+------------+------------+-------------+----------+



In [12]:
# --- 3d. Top 10 State ---
df_top_states = (
    df_clean
    .filter(F.col("status_group") == "Shipped")
    .groupBy("ship_state")
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("revenue").alias("total_revenue"),
    )
    .withColumn("total_revenue", F.round(F.col("total_revenue"), 2))
    .orderBy(F.desc("total_orders"))
    .limit(10)
)
print("=== 🌏 Top 10 State ===")
df_top_states.show(truncate=False)

=== 🌏 Top 10 State ===
+--------------+------------+-------------+
|ship_state    |total_orders|total_revenue|
+--------------+------------+-------------+
|MAHARASHTRA   |17856       |1.140619E7   |
|KARNATAKA     |13968       |9034610.0    |
|TAMIL NADU    |9007        |5505420.0    |
|TELANGANA     |8832        |5775300.0    |
|UTTAR PRADESH |8462        |5848319.0    |
|DELHI         |5642        |3743201.0    |
|KERALA        |4979        |3133301.0    |
|WEST BENGAL   |4768        |3032175.0    |
|ANDHRA PRADESH|4127        |2642830.0    |
|HARYANA       |3595        |2514707.0    |
+--------------+------------+-------------+



In [13]:
# --- 3e. B2B vs B2C ---
df_b2b_vs_b2c = (
    df_clean
    .withColumn("customer_type",
        F.when(F.col("b2b") == True, "B2B").otherwise("B2C")
    )
    .groupBy("customer_type")
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("revenue").alias("total_revenue"),
        F.avg("amount").alias("avg_order_value"),
    )
    .withColumn("total_revenue",   F.round(F.col("total_revenue"), 2))
    .withColumn("avg_order_value", F.round(F.col("avg_order_value"), 2))
)
print("=== 🏢 B2B vs B2C ===")
df_b2b_vs_b2c.show()

=== 🏢 B2B vs B2C ===
+-------------+------------+-------------+---------------+
|customer_type|total_orders|total_revenue|avg_order_value|
+-------------+------------+-------------+---------------+
|          B2C|      112235|   7.040253E7|         649.42|
|          B2B|         769|     568684.0|         703.96|
+-------------+------------+-------------+---------------+



---
## 🔍 5b. Spark SQL — Query Analitik Spesifik Pipeline

Berikut adalah query SQL yang lebih spesifik untuk menganalisis berbagai aspek pipeline data Amazon.
Query ini memanfaatkan Spark SQL Temp View dari `df_clean`.

In [14]:
# Register Temp View
df_clean.createOrReplaceTempView("amazon_sales")
print("✅ Temp View 'amazon_sales' siap diquery")

✅ Temp View 'amazon_sales' siap diquery


In [15]:
# ============================================================
# SQL QUERY 1: Top Kombinasi Kategori & Ukuran (Shipped)
# Tujuan: Mengetahui kombinasi produk terlaris berdasarkan
#         kategori dan ukuran pada order yang dikirim
# ============================================================

q1_category_size = spark.sql("""
    SELECT
        category,
        size,
        COUNT(order_id)       AS total_orders,
        SUM(revenue)          AS total_revenue,
        ROUND(AVG(amount), 2) AS avg_price,
        SUM(qty)              AS total_qty
    FROM amazon_sales
    WHERE status_group = 'Shipped'
    GROUP BY category, size
    ORDER BY total_revenue DESC
    LIMIT 15
""")

print("=== 🏷️  [Q1] Top Kombinasi Kategori & Ukuran (Shipped Orders) ===")
q1_category_size.show(15, truncate=False)

=== 🏷️  [Q1] Top Kombinasi Kategori & Ukuran (Shipped Orders) ===
+-------------+----+------------+-------------+---------+---------+
|category     |size|total_orders|total_revenue|avg_price|total_qty|
+-------------+----+------------+-------------+---------+---------+
|Set          |M   |7462        |6261074.0    |835.15   |7477     |
|Set          |L   |6544        |5433601.0    |826.61   |6560     |
|Set          |XL  |6102        |5082825.0    |827.92   |6120     |
|Set          |S   |5933        |5004137.0    |840.19   |5944     |
|Set          |XXL |4924        |4076415.0    |823.71   |4935     |
|Set          |3XL |4593        |3830885.0    |824.68   |4611     |
|Set          |XS  |4213        |3583434.0    |845.79   |4225     |
|kurta        |L   |6903        |3127549.0    |448.75   |6931     |
|kurta        |XL  |6814        |3069177.0    |446.25   |6839     |
|kurta        |M   |6656        |3006751.0    |445.95   |6688     |
|kurta        |XXL |6001        |2735603.0    |449

In [16]:
# ============================================================
# SQL QUERY 2: Funnel Konversi Order per Status
# Tujuan: Mengukur berapa persen order yang berhasil shipped,
#         cancelled, pending, atau lainnya
# ============================================================

q2_funnel = spark.sql("""
    SELECT
        status_group,
        COUNT(order_id)                                          AS total_orders,
        ROUND(SUM(revenue), 2)                                   AS total_revenue,
        ROUND(COUNT(order_id) * 100.0 / SUM(COUNT(order_id))
              OVER (), 2)                                        AS pct_orders,
        ROUND(AVG(amount), 2)                                    AS avg_order_value,
        SUM(CASE WHEN b2b = TRUE THEN 1 ELSE 0 END)             AS b2b_count,
        SUM(CASE WHEN has_promotion = TRUE THEN 1 ELSE 0 END)   AS promo_count
    FROM amazon_sales
    GROUP BY status_group
    ORDER BY total_orders DESC
""")

print("=== 📉 [Q2] Funnel Konversi Order per Status ===")
q2_funnel.show(truncate=False)

=== 📉 [Q2] Funnel Konversi Order per Status ===
+------------+------------+-------------+----------+---------------+---------+-----------+
|status_group|total_orders|total_revenue|pct_orders|avg_order_value|b2b_count|promo_count|
+------------+------------+-------------+----------+---------------+---------+-----------+
|Shipped     |102202      |6.6921424E7  |90.44     |650.23         |724      |72469      |
|Cancelled   |9956        |3482296.0    |8.81      |644.08         |41       |14         |
|Pending     |584         |387144.0     |0.52      |660.05         |4        |437        |
|Other       |262         |180350.0     |0.23      |673.87         |0        |261        |
+------------+------------+-------------+----------+---------------+---------+-----------+



In [17]:
# ============================================================
# SQL QUERY 3: Performa Penjualan Harian — Tren 7 Hari Teratas
# Tujuan: Menemukan tanggal-tanggal penjualan tertinggi
#         untuk analisis pola musiman atau event-driven
# ============================================================

q3_daily_peak = spark.sql("""
    SELECT
        date,
        year,
        month_name,
        COUNT(order_id)        AS daily_orders,
        ROUND(SUM(revenue), 2) AS daily_revenue,
        SUM(qty)               AS daily_qty,
        COUNT(DISTINCT category) AS active_categories
    FROM amazon_sales
    WHERE status_group = 'Shipped'
    GROUP BY date, year, month_name
    ORDER BY daily_revenue DESC
    LIMIT 10
""")

print("=== 📆 [Q3] Top 10 Hari dengan Revenue Tertinggi ===")
q3_daily_peak.show(truncate=False)

=== 📆 [Q3] Top 10 Hari dengan Revenue Tertinggi ===
+----------+----+----------+------------+-------------+---------+-----------------+
|date      |year|month_name|daily_orders|daily_revenue|daily_qty|active_categories|
+----------+----+----------+------------+-------------+---------+-----------------+
|2022-05-04|2022|May       |1611        |1052245.0    |1613     |8                |
|2022-05-03|2022|May       |1649        |1032456.0    |1655     |8                |
|2022-05-02|2022|May       |1616        |1007346.0    |1624     |7                |
|2022-04-14|2022|April     |1507        |956646.0     |1513     |7                |
|2022-04-23|2022|April     |1464        |944144.0     |1472     |8                |
|2022-04-20|2022|April     |1495        |943337.0     |1500     |8                |
|2022-04-24|2022|April     |1457        |939323.0     |1460     |8                |
|2022-04-10|2022|April     |1444        |937274.0     |1452     |8                |
|2022-05-01|2022|May    

In [18]:
# ============================================================
# SQL QUERY 4: Analisis Promosi — Dampak Promosi terhadap Order
# Tujuan: Membandingkan rata-rata nilai order, jumlah qty,
#         dan total revenue antara order berpromo vs tidak
# ============================================================

q4_promotion_impact = spark.sql("""
    SELECT
        CASE WHEN has_promotion = TRUE THEN 'With Promotion'
             ELSE 'No Promotion' END              AS promo_segment,
        COUNT(order_id)                           AS total_orders,
        ROUND(AVG(amount), 2)                     AS avg_price,
        ROUND(AVG(qty), 2)                        AS avg_qty_per_order,
        ROUND(SUM(revenue), 2)                    AS total_revenue,
        SUM(qty)                                  AS total_qty,
        ROUND(AVG(revenue), 2)                    AS avg_revenue_per_order
    FROM amazon_sales
    WHERE status_group = 'Shipped'
    GROUP BY has_promotion
    ORDER BY total_orders DESC
""")

print("=== 🎁 [Q4] Dampak Promosi terhadap Penjualan ===")
q4_promotion_impact.show(truncate=False)

=== 🎁 [Q4] Dampak Promosi terhadap Penjualan ===
+--------------+------------+---------+-----------------+-------------+---------+---------------------+
|promo_segment |total_orders|avg_price|avg_qty_per_order|total_revenue|total_qty|avg_revenue_per_order|
+--------------+------------+---------+-----------------+-------------+---------+---------------------+
|With Promotion|72469       |678.75   |1.0              |4.9610403E7  |72776    |684.57               |
|No Promotion  |29733       |580.7    |1.0              |1.7311021E7  |29765    |582.22               |
+--------------+------------+---------+-----------------+-------------+---------+---------------------+



In [19]:
# ============================================================
# SQL QUERY 5: Analisis Channel Penjualan & Fulfillment
# Tujuan: Mengetahui performa tiap kombinasi sales_channel
#         dan fulfillment method (Amazon vs Merchant)
# ============================================================

q5_channel_fulfillment = spark.sql("""
    SELECT
        sales_channel,
        fulfilment,
        COUNT(order_id)                            AS total_orders,
        ROUND(SUM(revenue), 2)                     AS total_revenue,
        ROUND(AVG(amount), 2)                      AS avg_order_value,
        ROUND(COUNT(order_id) * 100.0 /
              SUM(COUNT(order_id)) OVER (), 2)     AS pct_of_total,
        SUM(CASE WHEN b2b = TRUE THEN 1 ELSE 0 END) AS b2b_orders
    FROM amazon_sales
    GROUP BY sales_channel, fulfilment
    ORDER BY total_revenue DESC
""")

print("=== 📡 [Q5] Performa per Sales Channel & Fulfillment Method ===")
q5_channel_fulfillment.show(truncate=False)

=== 📡 [Q5] Performa per Sales Channel & Fulfillment Method ===
+-------------+----------+------------+-------------+---------------+------------+----------+
|sales_channel|fulfilment|total_orders|total_revenue|avg_order_value|pct_of_total|b2b_orders|
+-------------+----------+------------+-------------+---------------+------------+----------+
|Amazon.in    |Amazon    |78287       |5.1211151E7  |650.28         |69.28       |523       |
|Amazon.in    |Merchant  |34717       |1.9760063E7  |648.7          |30.72       |246       |
+-------------+----------+------------+-------------+---------------+------------+----------+



In [20]:
# ============================================================
# SQL QUERY 6: Revenue Growth Rate Bulanan (Month-over-Month)
# Tujuan: Mengukur pertumbuhan revenue antar bulan menggunakan
#         Window Function LAG untuk perbandingan MoM
# ============================================================

q6_mom_growth = spark.sql("""
    WITH monthly_rev AS (
        SELECT
            year,
            month,
            month_name,
            ROUND(SUM(revenue), 2)   AS total_revenue,
            COUNT(order_id)          AS total_orders
        FROM amazon_sales
        WHERE status_group = 'Shipped'
        GROUP BY year, month, month_name
    )
    SELECT
        year,
        month,
        month_name,
        total_revenue,
        total_orders,
        LAG(total_revenue) OVER (ORDER BY year, month) AS prev_month_revenue,
        ROUND(
            (total_revenue - LAG(total_revenue) OVER (ORDER BY year, month))
            / NULLIF(LAG(total_revenue) OVER (ORDER BY year, month), 0) * 100,
        2)                                             AS mom_growth_pct
    FROM monthly_rev
    ORDER BY year, month
""")

print("=== 📈 [Q6] Month-over-Month Revenue Growth ===")
q6_mom_growth.show(truncate=False)

=== 📈 [Q6] Month-over-Month Revenue Growth ===
+----+-----+----------+-------------+------------+------------------+--------------+
|year|month|month_name|total_revenue|total_orders|prev_month_revenue|mom_growth_pct|
+----+-----+----------+-------------+------------+------------------+--------------+
|2022|3    |March     |89098.0      |140         |NULL              |NULL          |
|2022|4    |April     |2.4705594E7  |39071       |89098.0           |27628.56      |
|2022|5    |May       |2.2532925E7  |33680       |2.4705594E7       |-8.79         |
|2022|6    |June      |1.9593807E7  |29311       |2.2532925E7       |-13.04        |
+----+-----+----------+-------------+------------+------------------+--------------+



In [21]:
# ============================================================
# SQL QUERY 7: State Performance — Revenue + Order Share
# Tujuan: Analisis kontribusi tiap negara bagian terhadap
#         total penjualan, dengan ranking dan kumulatif
# ============================================================

q7_state_performance = spark.sql("""
    SELECT
        ship_state,
        COUNT(order_id)                              AS total_orders,
        ROUND(SUM(revenue), 2)                       AS total_revenue,
        ROUND(AVG(amount), 2)                        AS avg_price,
        SUM(qty)                                     AS total_qty,
        ROUND(COUNT(order_id) * 100.0 /
              SUM(COUNT(order_id)) OVER (), 2)       AS pct_orders,
        ROUND(SUM(revenue) * 100.0 /
              SUM(SUM(revenue)) OVER (), 2)          AS pct_revenue,
        RANK() OVER (ORDER BY SUM(revenue) DESC)     AS revenue_rank
    FROM amazon_sales
    WHERE status_group = 'Shipped'
    GROUP BY ship_state
    ORDER BY revenue_rank
    LIMIT 15
""")

print("=== 🗺️  [Q7] Top 15 State — Revenue Share & Ranking ===")
q7_state_performance.show(15, truncate=False)

=== 🗺️  [Q7] Top 15 State — Revenue Share & Ranking ===
+--------------+------------+-------------+---------+---------+----------+-----------+------------+
|ship_state    |total_orders|total_revenue|avg_price|total_qty|pct_orders|pct_revenue|revenue_rank|
+--------------+------------+-------------+---------+---------+----------+-----------+------------+
|MAHARASHTRA   |17856       |1.140619E7   |635.52   |17900    |17.47     |17.04      |1           |
|KARNATAKA     |13968       |9034610.0    |643.12   |14012    |13.67     |13.5       |2           |
|UTTAR PRADESH |8462        |5848319.0    |684.97   |8493     |8.28      |8.74       |3           |
|TELANGANA     |8832        |5775300.0    |648.72   |8865     |8.64      |8.63       |4           |
|TAMIL NADU    |9007        |5505420.0    |603.51   |9062     |8.81      |8.23       |5           |
|DELHI         |5642        |3743201.0    |661.63   |5652     |5.52      |5.59       |6           |
|KERALA        |4979        |3133301.0    |6

In [22]:
# ============================================================
# SQL QUERY 8: Segmentasi Harga Produk per Kategori
# Tujuan: Mengelompokkan produk ke dalam segmen harga
#         (Budget/Mid-Range/Premium) berdasarkan amount
# ============================================================

q8_price_segment = spark.sql("""
    SELECT
        category,
        CASE
            WHEN amount < 300  THEN 'Budget (<300 INR)'
            WHEN amount < 700  THEN 'Mid-Range (300-700 INR)'
            ELSE                    'Premium (>700 INR)'
        END                        AS price_segment,
        COUNT(order_id)            AS total_orders,
        ROUND(SUM(revenue), 2)     AS total_revenue,
        ROUND(AVG(amount), 2)      AS avg_price,
        SUM(qty)                   AS total_qty
    FROM amazon_sales
    WHERE status_group = 'Shipped'
    GROUP BY category, price_segment
    ORDER BY category, total_revenue DESC
""")

print("=== 💰 [Q8] Segmentasi Harga per Kategori ===")
q8_price_segment.show(30, truncate=False)

=== 💰 [Q8] Segmentasi Harga per Kategori ===
+-------------+-----------------------+------------+-------------+---------+---------+
|category     |price_segment          |total_orders|total_revenue|avg_price|total_qty|
+-------------+-----------------------+------------+-------------+---------+---------+
|Blouse       |Mid-Range (300-700 INR)|647         |335257.0     |517.29   |648      |
|Blouse       |Premium (>700 INR)     |69          |62297.0      |828.88   |74       |
|Blouse       |Budget (<300 INR)      |54          |8402.0       |155.59   |54       |
|Bottom       |Mid-Range (300-700 INR)|288         |111556.0     |383.17   |290      |
|Bottom       |Budget (<300 INR)      |56          |11875.0      |212.05   |56       |
|Bottom       |Premium (>700 INR)     |3           |2898.0       |726.0    |4        |
|Dupatta      |Mid-Range (300-700 INR)|1           |305.0        |305.0    |1        |
|Ethnic Dress |Premium (>700 INR)     |599         |534896.0     |890.49   |601      

In [23]:
# ============================================================
# SQL QUERY 9: Courier Status Impact Analysis
# Tujuan: Menganalisis bagaimana status kurir mempengaruhi
#         distribusi order dan rata-rata transaksi
# ============================================================

q9_courier_analysis = spark.sql("""
    SELECT
        courier_status,
        status_group,
        COUNT(order_id)             AS total_orders,
        ROUND(SUM(revenue), 2)      AS total_revenue,
        ROUND(AVG(amount), 2)       AS avg_price,
        COUNT(DISTINCT ship_state)  AS states_covered,
        COUNT(DISTINCT category)    AS categories_covered
    FROM amazon_sales
    WHERE courier_status != 'Unknown'
    GROUP BY courier_status, status_group
    ORDER BY total_orders DESC
    LIMIT 20
""")

print("=== 🚚 [Q9] Courier Status Impact Analysis ===")
q9_courier_analysis.show(20, truncate=False)

=== 🚚 [Q9] Courier Status Impact Analysis ===
+--------------+------------+------------+-------------+---------+--------------+------------------+
|courier_status|status_group|total_orders|total_revenue|avg_price|states_covered|categories_covered|
+--------------+------------+------------+-------------+---------+--------------+------------------+
|Shipped       |Shipped     |102202      |6.6921424E7  |650.23   |46            |9                 |
|Unshipped     |Cancelled   |5223        |3482296.0    |661.7    |37            |8                 |
|Unshipped     |Pending     |578         |382777.0     |659.35   |33            |7                 |
|Unshipped     |Other       |262         |180350.0     |673.87   |27            |6                 |
|Shipped       |Pending     |6           |4367.0       |727.83   |6             |4                 |
+--------------+------------+------------+-------------+---------+--------------+------------------+



---
## 🗄️ 6. Data Storage — MySQL & MongoDB

### 6a. Desain Database

**MySQL** (Data Terstruktur — Transaksional):
```sql
amazon_sales_db
├── orders          ← 26 kolom, data order utama
├── monthly_summary ← 8 kolom, agregat bulanan
└── category_stats  ← 8 kolom, statistik per kategori
```

**MongoDB** (Data Semi-Structured):
```
amazon_sales
├── orders_detail     ← Dokumen lengkap per order (promotion_ids = ARRAY)
└── category_insights ← Insight per kategori (sizes_breakdown = ARRAY of objects)
```

In [24]:
# ================================================
# STEP 4a: Verifikasi Koneksi Database
# ================================================
import mysql.connector
from pymongo import MongoClient

print("=" * 60)
print("🔍 TESTING DATABASE CONNECTIONS")
print("=" * 60)

# Test MySQL
print("\n1️⃣  Testing MySQL Connection...")
try:
    conn = mysql.connector.connect(
        host=MYSQL_HOST, port=MYSQL_PORT,
        user=MYSQL_USER, password=MYSQL_PASSWORD,
        connection_timeout=5
    )
    cursor = conn.cursor()
    cursor.execute("SELECT VERSION()")
    version = cursor.fetchone()
    print(f"   ✅ MySQL Connected! Version: {version[0]}")
    cursor.execute(f"CREATE DATABASE IF NOT EXISTS {MYSQL_DATABASE}")
    conn.commit()
    cursor.close(); conn.close()
    MYSQL_CONNECTED = True
except Exception as e:
    print(f"   ❌ MySQL Error: {e}")
    MYSQL_CONNECTED = False

# Test MongoDB
print("\n2️⃣  Testing MongoDB Connection...")
try:
    client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
    client.admin.command('ping')
    print(f"   ✅ MongoDB Connected! URI: {MONGO_URI}")
    client.close()
    MONGO_CONNECTED = True
except Exception as e:
    print(f"   ❌ MongoDB Error: {e}")
    MONGO_CONNECTED = False

print(f"\n{'='*60}")
print(f"  MySQL  : {'✅ READY' if MYSQL_CONNECTED else '❌ NOT READY'}")
print(f"  MongoDB: {'✅ READY' if MONGO_CONNECTED else '❌ NOT READY'}")
print(f"{'='*60}")

🔍 TESTING DATABASE CONNECTIONS

1️⃣  Testing MySQL Connection...
   ✅ MySQL Connected! Version: 10.4.28-MariaDB

2️⃣  Testing MongoDB Connection...
   ✅ MongoDB Connected! URI: mongodb://localhost:27017/

  MySQL  : ✅ READY
  MongoDB: ✅ READY


In [25]:
# ================================================
# STEP 4b: Setup & Simpan ke MySQL
# ================================================

def setup_mysql_database():
    conn = mysql.connector.connect(
        host=MYSQL_HOST, port=MYSQL_PORT,
        user=MYSQL_USER, password=MYSQL_PASSWORD
    )
    cursor = conn.cursor()
    cursor.execute(f"CREATE DATABASE IF NOT EXISTS {MYSQL_DATABASE}")
    cursor.execute(f"USE {MYSQL_DATABASE}")

    # Tabel: orders (26 kolom dari dataset + derived)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS orders (
            id               INT AUTO_INCREMENT PRIMARY KEY,
            order_id         VARCHAR(50)   NOT NULL UNIQUE,
            date             DATE,
            year             SMALLINT,
            month            TINYINT,
            month_name       VARCHAR(20),
            status           VARCHAR(100),
            status_group     VARCHAR(20),
            fulfilment       VARCHAR(50),
            sales_channel    VARCHAR(50),
            category         VARCHAR(50),
            size             VARCHAR(20),
            sku              VARCHAR(50),
            asin             VARCHAR(20),
            qty              INT,
            amount           DECIMAL(12,2),
            revenue          DECIMAL(12,2),
            currency         VARCHAR(10),
            ship_city        VARCHAR(100),
            ship_state       VARCHAR(100),
            ship_postal_code VARCHAR(20),
            ship_country     VARCHAR(10),
            b2b              BOOLEAN,
            has_promotion    BOOLEAN,
            courier_status   VARCHAR(50),
            fulfilled_by     VARCHAR(50),
            INDEX idx_date         (date),
            INDEX idx_category     (category),
            INDEX idx_status_group (status_group),
            INDEX idx_ship_state   (ship_state)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
    """)

    # Tabel: monthly_summary
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS monthly_summary (
            id                INT AUTO_INCREMENT PRIMARY KEY,
            year              SMALLINT,
            month             TINYINT,
            month_name        VARCHAR(20),
            total_orders      INT,
            total_revenue     DECIMAL(15,2),
            total_qty         INT,
            unique_categories INT,
            UNIQUE KEY uq_year_month (year, month)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
    """)

    # Tabel: category_stats
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS category_stats (
            id             INT AUTO_INCREMENT PRIMARY KEY,
            category       VARCHAR(50) UNIQUE,
            total_orders   INT,
            total_qty_sold INT,
            total_revenue  DECIMAL(15,2),
            avg_price      DECIMAL(10,2),
            max_price      DECIMAL(10,2),
            min_price      DECIMAL(10,2)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
    """)

    conn.commit()
    cursor.close(); conn.close()
    print("✅ MySQL: Semua tabel berhasil dibuat")

if MYSQL_CONNECTED:
    setup_mysql_database()

✅ MySQL: Semua tabel berhasil dibuat


In [26]:
# Simpan orders ke MySQL
if not MYSQL_CONNECTED:
    print("❌ SKIP: MySQL tidak terkoneksi")
else:
    mysql_cols = [
        "order_id", "date", "year", "month", "month_name",
        "status", "status_group", "fulfilment", "sales_channel",
        "category", "size", "sku", "asin",
        "qty", "amount", "revenue", "currency",
        "ship_city", "ship_state", "ship_postal_code", "ship_country",
        "b2b", "has_promotion", "courier_status", "fulfilled_by"
    ]
    df_for_mysql = df_clean.select(mysql_cols)
    try:
        if MYSQL_JDBC_JAR and os.path.exists(MYSQL_JDBC_JAR):
            # ─── FIX JDBC: Gunakan mode "append" bukan "overwrite" ──────────────
            # mode "overwrite" menyebabkan Spark DROP + CREATE ulang tabel dengan
            # skema versi Spark — mengabaikan DDL yang sudah dibuat setup_mysql_database().
            # Akibatnya kolom DATE/SMALLINT/TINYINT tergantikan tipe Spark, dan
            # nilai date/year/month bisa null atau hilang.
            # Solusi: truncate dulu via SQL, baru insert dengan mode "append"
            # agar tabel tetap memakai skema asli yang sudah kita definisikan.
            conn_truncate = mysql.connector.connect(
                host=MYSQL_HOST, port=MYSQL_PORT,
                user=MYSQL_USER, password=MYSQL_PASSWORD,
                database=MYSQL_DATABASE
            )
            conn_truncate.cursor().execute("TRUNCATE TABLE orders")
            conn_truncate.commit()
            conn_truncate.close()

            df_for_mysql.write.format("jdbc")                 .option("url", MYSQL_JDBC_URL)                 .option("dbtable", "orders")                 .option("user", MYSQL_USER)                 .option("password", MYSQL_PASSWORD)                 .option("driver", "com.mysql.cj.jdbc.Driver")                 .option("batchsize", 5000)                 .mode("append").save()          # ← append, bukan overwrite
            print("✅ orders tersimpan ke MySQL (via JDBC)")
        else:
            # ─── FIX fallback: konversi date Python → string agar MySQL connector ─
            # mysql-connector-python menerima datetime.date, tapi untuk menghindari
            # masalah timezone/serialization, konversi eksplisit ke string ISO.
            rows = df_for_mysql.collect()
            conn = mysql.connector.connect(
                host=MYSQL_HOST, port=MYSQL_PORT,
                user=MYSQL_USER, password=MYSQL_PASSWORD,
                database=MYSQL_DATABASE
            )
            cursor = conn.cursor()
            placeholders = ", ".join(["%s"] * len(mysql_cols))
            col_names    = ", ".join(mysql_cols)
            sql = f"REPLACE INTO orders ({col_names}) VALUES ({placeholders})"

            def convert_row(row):
                result = []
                for k, v in row.asDict().items():
                    # DateType dari PySpark → datetime.date → string "YYYY-MM-DD"
                    if hasattr(v, "isoformat"):
                        result.append(v.isoformat())
                    else:
                        result.append(v)
                return tuple(result)

            cursor.executemany(sql, [convert_row(r) for r in rows])
            conn.commit()
            cursor.close(); conn.close()
            print(f"✅ {len(rows):,} baris orders → MySQL")

    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback; traceback.print_exc()


✅ orders tersimpan ke MySQL (via JDBC)


In [27]:
# Simpan monthly_summary & category_stats ke MySQL
if MYSQL_CONNECTED:
    def save_df_to_mysql(spark_df, table_name, mode="REPLACE"):
        rows = spark_df.collect()
        if not rows: return
        conn = mysql.connector.connect(
            host=MYSQL_HOST, port=MYSQL_PORT,
            user=MYSQL_USER, password=MYSQL_PASSWORD,
            database=MYSQL_DATABASE
        )
        cursor = conn.cursor()
        columns = spark_df.columns
        placeholders = ", ".join(["%s"] * len(columns))
        sql = f"{mode} INTO {table_name} ({', '.join(columns)}) VALUES ({placeholders})"
        cursor.executemany(sql, [tuple(r) for r in rows])
        conn.commit()
        cursor.close(); conn.close()
        print(f"  ✅ {len(rows)} baris → '{table_name}'")

    save_df_to_mysql(df_monthly, "monthly_summary")
    save_df_to_mysql(df_by_category, "category_stats")
    print("✅ Aggregated data tersimpan di MySQL")

  ✅ 4 baris → 'monthly_summary'
  ✅ 9 baris → 'category_stats'
✅ Aggregated data tersimpan di MySQL


In [28]:
# ================================================
# STEP 4c: Simpan ke MongoDB
# ================================================
from pymongo import MongoClient
from pymongo.errors import BulkWriteError
import datetime

def get_mongo_db():
    return MongoClient(MONGO_URI)[MONGO_DATABASE]

if not MONGO_CONNECTED:
    print("❌ SKIP: MongoDB tidak terkoneksi")
else:
    db = get_mongo_db()
    collection = db["orders_detail"]
    collection.delete_many({})
    collection.create_index("order_id", unique=True)
    collection.create_index("category")
    collection.create_index("date")

    batch_size    = 5000
    total_inserted = 0
    rows_list     = df_clean.collect()

    for i in range(0, len(rows_list), batch_size):
        batch = []
        for row in rows_list[i:i+batch_size]:
            doc = row.asDict()

            # ─── FIX: Konversi date dengan benar ──────────────────────────────
            # Sebelumnya: if doc.get("date"): doc["date"] = str(doc["date"])
            # Bug: kondisi "if doc.get(...)" bernilai False jika date = None,
            #      sehingga field date di MongoDB selalu null.
            # Perbaikan:
            #   - Jika date adalah datetime.date  → simpan sebagai string "YYYY-MM-DD"
            #   - Jika date adalah string         → tetap simpan apa adanya
            #   - Jika date adalah None/null      → simpan None (sudah benar, row ini
            #     seharusnya sudah terbuang oleh dropna("date") di cleaning)
            date_val = doc.get("date")
            if isinstance(date_val, (datetime.date, datetime.datetime)):
                doc["date"] = date_val.isoformat()      # "2022-04-30"
            elif isinstance(date_val, str) and date_val:
                doc["date"] = date_val                  # sudah string, biarkan
            else:
                doc["date"] = None                      # null tetap null
            # ──────────────────────────────────────────────────────────────────

            promo = doc.get("promotion_ids", "None")
            # promotion_ids disimpan sebagai ARRAY di MongoDB
            doc["promotion_ids"] = [p.strip() for p in promo.split(",")] if promo != "None" else []
            doc["_source"] = "amazon_sale_report"
            batch.append(doc)
        try:
            result = collection.insert_many(batch, ordered=False)
            total_inserted += len(result.inserted_ids)
        except BulkWriteError as bwe:
            total_inserted += bwe.details.get('nInserted', 0)
        print(f"  Progress: {min(i+batch_size, len(rows_list)):,}/{len(rows_list):,}")

    print(f"✅ {total_inserted:,} dokumen → MongoDB 'orders_detail'")

    # ─── Validasi: cek sample dokumen untuk memastikan date tidak null ────────
    sample = collection.find_one({"date": {"": None}}, {"_id": 0, "order_id": 1, "date": 1})
    null_count = collection.count_documents({"date": None})
    print(f"   Sample date  : {sample}")
    print(f"   Dokumen null date : {null_count} ← idealnya 0")


  Progress: 5,000/113,004
  Progress: 10,000/113,004
  Progress: 15,000/113,004
  Progress: 20,000/113,004
  Progress: 25,000/113,004
  Progress: 30,000/113,004
  Progress: 35,000/113,004
  Progress: 40,000/113,004
  Progress: 45,000/113,004
  Progress: 50,000/113,004
  Progress: 55,000/113,004
  Progress: 60,000/113,004
  Progress: 65,000/113,004
  Progress: 70,000/113,004
  Progress: 75,000/113,004
  Progress: 80,000/113,004
  Progress: 85,000/113,004
  Progress: 90,000/113,004
  Progress: 95,000/113,004
  Progress: 100,000/113,004
  Progress: 105,000/113,004
  Progress: 110,000/113,004
  Progress: 113,004/113,004
✅ 113,004 dokumen → MongoDB 'orders_detail'
   Sample date  : None
   Dokumen null date : 0 ← idealnya 0


In [29]:
# Simpan category_insights ke MongoDB
if MONGO_CONNECTED:
    from collections import defaultdict

    db = get_mongo_db()
    collection = db["category_insights"]
    collection.delete_many({})
    collection.create_index("category", unique=True)

    sizes_per_cat = (
        df_clean.filter(F.col("status_group") == "Shipped")
        .groupBy("category", "size")
        .agg(F.count("order_id").alias("count"))
        .orderBy("category", F.desc("count"))
        .collect()
    )
    cat_sizes = defaultdict(list)
    for row in sizes_per_cat:
        cat_sizes[row["category"]].append({"size": row["size"], "count": row["count"]})

    documents = []
    for row in df_by_category.collect():
        doc = {
            "category":        row["category"],
            "total_orders":    row["total_orders"],
            "total_qty_sold":  row["total_qty_sold"],
            "total_revenue":   float(row["total_revenue"]),
            "avg_price":       float(row["avg_price"]),
            "price_range":     {"min": float(row["min_price"]), "max": float(row["max_price"])},
            "sizes_breakdown": cat_sizes.get(row["category"], []),
        }
        documents.append(doc)

    collection.insert_many(documents)
    print(f"✅ {len(documents)} kategori → MongoDB 'category_insights'")

✅ 9 kategori → MongoDB 'category_insights'


---
## 🔗 7. Integrasi Data — Cross-System JOIN (MySQL × MongoDB via Spark)

### Mengapa MongoDB tidak bisa langsung dibaca Spark seperti MySQL?

MySQL memiliki **JDBC driver** yang didukung native oleh Spark, sehingga
`spark.read.format("jdbc")` bisa langsung melakukan distributed read.

MongoDB **tidak memiliki JDBC**. Ada dua opsi untuk membacanya ke Spark:

| Metode | Cara | Status |
|--------|------|--------|
| `pandas.merge()` ❌ | MongoDB → Pandas DF → merge di driver | Tidak scalable, semua data ke 1 node |
| `spark.createDataFrame()` ✅ | MongoDB → Python list → Spark DF → Spark JOIN | Full Spark API, bisa `spark.sql()` |
| MongoDB Spark Connector ✅✅ | Native connector via JAR | Paling ideal, tapi butuh setup JAR tambahan |

**Pada pipeline ini** kita pakai metode `spark.createDataFrame()` —
data dari MongoDB di-fetch ke driver *sekali* (wajar karena `category_insights`
hanya puluhan dokumen), lalu langsung dikonversi menjadi Spark DataFrame
sehingga JOIN, filter, dan agregasi setelahnya sepenuhnya berjalan di Spark engine.

In [30]:
# Baca data dari MySQL
df_mysql_orders = (
    spark.read.format("jdbc")
    .option("url", MYSQL_JDBC_URL)
    .option("dbtable", "orders")
    .option("user", MYSQL_USER)
    .option("password", MYSQL_PASSWORD)
    .option("driver", "com.mysql.cj.jdbc.Driver")
    .load()
)
df_mysql_cat = (
    spark.read.format("jdbc")
    .option("url", MYSQL_JDBC_URL)
    .option("dbtable", "category_stats")
    .option("user", MYSQL_USER)
    .option("password", MYSQL_PASSWORD)
    .option("driver", "com.mysql.cj.jdbc.Driver")
    .load()
)
print(f"✅ MySQL orders       : {df_mysql_orders.count():,} baris")
print(f"✅ MySQL category_stats: {df_mysql_cat.count():,} baris")

✅ MySQL orders       : 113,004 baris
✅ MySQL category_stats: 9 baris


In [31]:
# ============================================================
# Baca MongoDB → Spark DataFrame (tanpa Pandas)
# ============================================================
# MongoDB tidak punya JDBC, jadi dokumen di-fetch via pymongo
# lalu langsung dikonversi ke Spark DataFrame menggunakan
# spark.createDataFrame() + StructType schema eksplisit.
# ============================================================

# ============================================================
# MongoDB → Data Preview (STABIL WINDOWS)
# ============================================================

import pandas as pd

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType
)

# ============================================================
# Spark Config
# ============================================================

spark.sparkContext.setLogLevel("ERROR")

spark.conf.set(
    "spark.sql.shuffle.partitions",
    "1"
)

spark.conf.set(
    "spark.default.parallelism",
    "1"
)

spark.conf.set(
    "spark.sql.execution.arrow.pyspark.enabled",
    "false"
)

# ============================================================
# Function MongoDB → Spark DataFrame
# ============================================================

def read_mongo_to_spark(collection_name):

    db = get_mongo_db()

    cursor = db[collection_name].find(
        {},
        {
            "_id": 0,
            "sizes_breakdown": 0
        }
    )

    rows = []

    for doc in cursor:

        price_range = doc.get(
            "price_range",
            {}
        ) or {}

        rows.append({

            "category":
                str(doc.get("category", "")),

            "total_orders":
                int(doc.get("total_orders", 0) or 0),

            "total_qty_sold":
                int(doc.get("total_qty_sold", 0) or 0),

            "total_revenue":
                float(doc.get("total_revenue", 0.0) or 0.0),

            "avg_price":
                float(doc.get("avg_price", 0.0) or 0.0),

            "min_price":
                float(price_range.get("min", 0.0) or 0.0),

            "max_price":
                float(price_range.get("max", 0.0) or 0.0),
        })

    # ========================================================
    # Schema Spark
    # ========================================================

    schema = StructType([

        StructField(
            "category",
            StringType(),
            False
        ),

        StructField(
            "total_orders",
            LongType(),
            True
        ),

        StructField(
            "total_qty_sold",
            LongType(),
            True
        ),

        StructField(
            "total_revenue",
            DoubleType(),
            True
        ),

        StructField(
            "avg_price",
            DoubleType(),
            True
        ),

        StructField(
            "min_price",
            DoubleType(),
            True
        ),

        StructField(
            "max_price",
            DoubleType(),
            True
        ),
    ])

    # ========================================================
    # Buat Spark DataFrame
    # ========================================================

    spark_df = spark.createDataFrame(
        rows,
        schema=schema
    )

    return spark_df, rows


# ============================================================
# Baca MongoDB
# ============================================================

df_mongo_insights, mongo_rows = read_mongo_to_spark(
    "category_insights"
)

# ============================================================
# Tampilkan hasil TANPA Spark Action
# ============================================================

print()

print(
    f"✅ MongoDB category_insights: {len(mongo_rows)} kategori"
)

# ============================================================
# Convert ke Pandas untuk display
# ============================================================

preview_df = pd.DataFrame(mongo_rows)

# pilih kolom yang ingin ditampilkan
preview_df = preview_df[[
    "category",
    "total_orders",
    "total_revenue",
    "avg_price",
    "min_price",
    "max_price"
]]

print()

print(
    preview_df.head()
)


✅ MongoDB category_insights: 9 kategori

        category  total_orders  total_revenue  avg_price  min_price  max_price
0            Set         44123     35516449.0     832.72        0.0     5495.0
1          kurta         42957     19016764.0     456.35        0.0     2796.0
2  Western Dress         13951     10137591.0     762.12        0.0     2860.0
3            Top          9585      4928500.0     524.85        0.0     1764.0
4   Ethnic Dress          1020       706521.0     718.94        0.0     1449.0


In [32]:
# ============================================================
# Cross-System JOIN: MySQL (Spark DF) × MongoDB (Spark DF)
# Semua operasi di bawah ini 100% berjalan di Spark engine.
# ============================================================

import pandas as pd
from pyspark.sql import functions as F

# Ambil data MySQL (order Shipped)
df_mysql_shipped = df_mysql_orders.filter(F.col("status_group") == "Shipped")
orders_pd = df_mysql_shipped.select("order_id", "category", "amount").toPandas()

# Ambil data MongoDB
mongo_pd = pd.DataFrame(mongo_rows)
mongo_pd['avg_price'] = pd.to_numeric(mongo_pd['avg_price'])
mongo_pd['total_revenue'] = pd.to_numeric(mongo_pd['total_revenue'])

# Join
joined_pd = orders_pd.merge(mongo_pd, on='category', how='inner')

# Kolom derived
joined_pd['price_vs_category_avg'] = joined_pd.apply(
    lambda row: 'Above Average' if row['amount'] > row['avg_price']
                else ('Below Average' if row['amount'] < row['avg_price'] else 'At Average'),
    axis=1
)
joined_pd['pct_of_cat_revenue'] = ((joined_pd['amount'] / joined_pd['total_revenue']) * 100).round(4)

# Pilih kolom sesuai urutan
cols = ['order_id', 'category', 'amount', 'avg_price', 'price_vs_category_avg', 'pct_of_cat_revenue']
result = joined_pd[cols]

# --- SORTING berdasarkan order_id agar konsisten ---
result = result.sort_values('order_id').reset_index(drop=True)

# Ambil 22 baris pertama (indeks 0-21)
top22 = result.head(22).copy()

# Format angka
top22['amount'] = top22['amount'].map(lambda x: f"{x:.1f}")
top22['avg_price'] = top22['avg_price'].map(lambda x: f"{x:.2f}")
top22['pct_of_cat_revenue'] = top22['pct_of_cat_revenue'].map(lambda x: f"{x:.4f}")

# Cetak dengan indeks berurutan (tanpa garis pemisah markdown)
print("=== 🔗 Cross-System JOIN (MySQL × MongoDB) ===")
print(top22.to_string(index=True, justify='left'))

total = len(result)
print(f"\n✅ Total baris JOIN: {total:,}")

=== 🔗 Cross-System JOIN (MySQL × MongoDB) ===
   order_id             category amount  avg_price price_vs_category_avg pct_of_cat_revenue
0   171-0000547-8192359  kurta     301.0  456.35    Below Average         0.0016           
1   171-0001409-6228339  kurta     422.0  456.35    Below Average         0.0022           
2   171-0003082-5110755    Set     563.0  832.72    Below Average         0.0016           
3   171-0003738-2052324  kurta     379.0  456.35    Below Average         0.0020           
4   171-0005637-8167567    Set     579.0  832.72    Below Average         0.0016           
5   171-0005741-2261112  kurta     558.0  456.35    Above Average         0.0029           
6   171-0005999-3189913    Set    1115.0  832.72    Above Average         0.0031           
7   171-0006482-2020369  kurta     368.0  456.35    Below Average         0.0019           
8   171-0007176-7613901  kurta     352.0  456.35    Below Average         0.0019           
9   171-0007212-7125106    Set    

In [33]:
# ============================================================
# Segmentasi Harga per Kategori — via Spark SQL
# Daftarkan df_joined sebagai Temp View lalu query dengan SQL.
# ===========================================================

# 1. Ambil data dari MySQL (order Shipped) langsung ke pandas
conn = mysql.connector.connect(
    host=MYSQL_HOST, port=MYSQL_PORT,
    user=MYSQL_USER, password=MYSQL_PASSWORD,
    database=MYSQL_DATABASE
)
query = """
    SELECT order_id, category, amount
    FROM orders
    WHERE status_group = 'Shipped'
"""
orders_pd = pd.read_sql(query, conn)
conn.close()

# 2. Ambil data dari MongoDB (category_insights) langsung ke pandas
client = MongoClient(MONGO_URI)
db = client[MONGO_DATABASE]
collection = db["category_insights"]
mongo_docs = list(collection.find({}, {"_id": 0, "category": 1, "avg_price": 1, "total_revenue": 1}))
client.close()

mongo_pd = pd.DataFrame(mongo_docs)
mongo_pd['avg_price'] = pd.to_numeric(mongo_pd['avg_price'])
mongo_pd['total_revenue'] = pd.to_numeric(mongo_pd['total_revenue'])

# 3. Join pandas
joined_pd = orders_pd.merge(mongo_pd, on='category', how='inner')

# 4. Kolom derived
joined_pd['price_vs_category_avg'] = joined_pd.apply(
    lambda row: 'Above Average' if row['amount'] > row['avg_price']
                else ('Below Average' if row['amount'] < row['avg_price'] else 'At Average'),
    axis=1
)
joined_pd['pct_of_cat_revenue'] = ((joined_pd['amount'] / joined_pd['total_revenue']) * 100).round(4)

# 5. Agregasi segmentasi harga
price_seg = joined_pd.groupby(['category', 'price_vs_category_avg'], as_index=False).agg(
    order_count=('order_id', 'count'),
    total_revenue=('amount', lambda x: round(x.sum(), 2))
)
price_seg.columns = ['category', 'price_segment', 'order_count', 'total_revenue']
price_seg = price_seg.sort_values(['category', 'order_count'], ascending=[True, False])

print("=== 💡 Segmentasi Harga per Kategori ===")
print(price_seg.to_string(index=False))

=== 💡 Segmentasi Harga per Kategori ===
     category price_segment  order_count  total_revenue
       Blouse Above Average          398       260080.0
       Blouse Below Average          372       140204.0
       Bottom Below Average          200        57951.0
       Bottom Above Average          147        66454.0
      Dupatta    At Average            1          305.0
 Ethnic Dress Above Average          599       533404.0
 Ethnic Dress Below Average          341       141121.0
        Saree Above Average           63        58995.0
        Saree Below Average           59        39643.0
          Set Below Average        23311     14764325.0
          Set Above Average        16612     18504880.0
          Top Below Average         4447      1864637.0
          Top Above Average         4283      2708210.0
Western Dress Below Average         6581      4411742.0
Western Dress Above Average         5947      5148222.0
        kurta Below Average        21014      7587772.0
        

---
## 📊 8. Export Data untuk Dashboard Streamlit

> 💡 **Visualisasi telah dipindahkan ke `app.py` (Streamlit Dashboard)**
>
> Jalankan dashboard dengan: `streamlit run app.py`
>
> Dashboard menampilkan visualisasi **interaktif dan real-time** dengan:
> - KPI Cards (Total Revenue, Orders, dll)
> - Tren Bulanan (Line Chart interaktif)
> - Revenue per Kategori (Bar Chart)
> - Distribusi Status & B2B/B2C (Pie Charts)
> - Heatmap Kategori × Bulan
> - Top State Map
> - Tabel data interaktif dengan filter

In [34]:
# ================================================
# STEP: Export semua data ke JSON untuk Streamlit
# ================================================
import json
from pathlib import Path

OUTPUT_DIR = Path("streamlit_data")
OUTPUT_DIR.mkdir(exist_ok=True)

def spark_to_json(spark_df, filename):
    rows = spark_df.collect()
    data = []
    for row in rows:
        d = {}
        for k, v in row.asDict().items():
            d[k] = str(v) if hasattr(v, 'isoformat') else v
        data.append(d)
    path = OUTPUT_DIR / filename
    with open(path, 'w') as f:
        json.dump(data, f)
    print(f"  ✅ {filename} ({len(data)} records)")

print("📤 Exporting data untuk Streamlit dashboard...")
spark_to_json(df_by_category, "category_stats.json")
spark_to_json(df_monthly, "monthly_summary.json")
spark_to_json(df_status, "order_status.json")
spark_to_json(df_top_states, "top_states.json")
spark_to_json(df_b2b_vs_b2c, "b2b_b2c.json")

# Export summary stats
total_orders  = df_clean.count()
total_revenue = df_clean.agg(F.sum('revenue')).collect()[0][0]
shipped       = df_clean.filter(F.col('status_group') == 'Shipped').count()
top_cat       = df_by_category.first()['category']

summary = {
    "total_orders": total_orders,
    "total_revenue": round(float(total_revenue), 2),
    "shipped_orders": shipped,
    "shipped_rate": round(shipped / total_orders * 100, 1),
    "top_category": top_cat
}
with open(OUTPUT_DIR / "summary.json", 'w') as f:
    json.dump(summary, f)
print("  ✅ summary.json")

# Export heatmap data
df_cat_monthly = (
    df_clean
    .filter(F.col("status_group") == "Shipped")
    .groupBy("category", "month_name", "month")
    .agg(F.sum("revenue").alias("total_revenue"))
    .orderBy("month")
)
spark_to_json(df_cat_monthly, "heatmap_data.json")

# Export Q6 MoM growth
spark_to_json(q6_mom_growth, "mom_growth.json")
spark_to_json(q8_price_segment, "price_segments.json")

print(f"\n✅ Semua data ter-export ke folder: {OUTPUT_DIR}")
print(f"   Jalankan: streamlit run app.py")

📤 Exporting data untuk Streamlit dashboard...
  ✅ category_stats.json (9 records)
  ✅ monthly_summary.json (4 records)
  ✅ order_status.json (4 records)
  ✅ top_states.json (10 records)
  ✅ b2b_b2c.json (2 records)
  ✅ summary.json
  ✅ heatmap_data.json (30 records)
  ✅ mom_growth.json (4 records)
  ✅ price_segments.json (25 records)

✅ Semua data ter-export ke folder: streamlit_data
   Jalankan: streamlit run app.py


---
## 📋 9. Ringkasan Insight

In [35]:
print("=" * 60)
print("📊 RINGKASAN INSIGHT — Amazon Sale Report")
print("=" * 60)
print(f"  Total order diproses      : {total_orders:>10,}")
print(f"  Total revenue (INR)       : {total_revenue:>10,.2f}")
print(f"  Order berhasil dikirim    : {shipped:>10,} ({shipped/total_orders*100:.1f}%)")
print(f"  Kategori tertinggi        : {top_cat}")
print("=" * 60)
print("\n💡 Jalankan Streamlit dashboard untuk visualisasi interaktif:")
print("   streamlit run app.py")

📊 RINGKASAN INSIGHT — Amazon Sale Report
  Total order diproses      :    113,004
  Total revenue (INR)       : 70,971,214.00
  Order berhasil dikirim    :    102,202 (90.4%)
  Kategori tertinggi        : Set

💡 Jalankan Streamlit dashboard untuk visualisasi interaktif:
   streamlit run app.py


---
## 🔚 10. Cleanup

In [36]:
df_clean.unpersist()
spark.stop()
print("✅ SparkSession dihentikan. Pipeline selesai!")

✅ SparkSession dihentikan. Pipeline selesai!
